[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Python from the Start](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)

# Scope


## What you will be able to do

Say which names a function can see, which it can change, and why a value you passed in sometimes
comes back altered and sometimes does not. You will also be able to read the error Python gives
when a function assigns to a name it only meant to read.


## The idea

### The problem

The **Functions** notebook gave you a way to name a piece of code and run it with different
values. It left an obvious question unanswered: inside that function, which names actually
exist?

Two things happen that look inconsistent until you know the rule.

A function can **read** a name defined outside it, without declaring anything. But a function that
tries to **assign** to that same name gets an error that does not sound like an explanation:
`cannot access local variable 'count' where it is not associated with a value`.

And a list handed to a function sometimes comes back changed and sometimes does not, depending
on what the function did to it, with nothing in the call to tell you which.

Neither is arbitrary. Both follow from one rule.

### What scope is

> **Scope** is the region of code in which a name is visible. Every function call creates a new
> **local scope**. Names assigned inside a function belong to that scope and disappear when the
> call ends.
>
> When Python looks up a name, it searches outward in a fixed order: **local** first, then any
> **enclosing** function, then the **global** names of the notebook, then Python's **built-ins**.
> The first match wins.

That order is worth remembering as four steps outward. It explains lookups completely.

### The rule that explains assignment

Looking a name up searches outward. **Assigning to a name does not.** An assignment always
creates or changes a name in the current scope, and never reaches out to touch one further up.

That single asymmetry explains everything in this notebook: lookup searches outward,
assignment stays local.

It is also why the error message sounds strange. If a function assigns to `count` anywhere in
its body, Python decides `count` is local for the entire function, including the lines above the
assignment. Reading it before that point is then reading a local name that has not been given a
value yet, which is exactly what the message says.

### Where you will meet this

The moment a program is more than one function, which is immediately. It explains why a
function seemed to do nothing, why a value did not update, and why a name you were sure existed
was reported missing.

It also explains one thing already in use: the `key=lambda` argument in the **Lists** and
**Functions** notebooks works because a small function can see the names around it.

### What this notebook covers

- Local names, and how completely they vanish when the call ends
- Reading an outer name, which works, and assigning to one, which does not
- `global`, and why it is a last resort rather than a fix
- **Mutating** a value you were passed against **rebinding** the name, which is the difference
  between changing the caller's data and not
- Shadowing, including the built-in name that stops working when you reuse it
- Nested functions, `nonlocal`, and **closures**
- Four errors, one of which does not raise

Start with a name that exists only inside a call.


In [1]:
def compute():
    answer = 42
    return "done"

print(compute())
print("answer" in dir())     # is there an 'answer' out here?


done
False


`answer` was created, used, and destroyed when the call ended. Nothing outside can see it, which
is what makes functions safe to write without checking whether a name is already taken.


## Setup

Nothing to import.


In [2]:
# This notebook uses no libraries.
print("Ready.")


Ready.


## Worked examples

### Reading outward works

A function can use a name defined outside it.


In [3]:
tax_rate = 0.08

def with_tax(amount):
    return round(amount * (1 + tax_rate), 2)

print(with_tax(100))


108.0


`tax_rate` is not a parameter and not local. Python did not find it locally, looked outward, and
found it among the notebook's names.

Convenient, and worth using sparingly. A function that quietly depends on outside names is
harder to move, harder to test, and breaks when the name changes. Passing it in as a parameter
says what the function needs.

### Assigning outward does not

The same name, with an assignment added, behaves completely differently.


In [4]:
counter = 0

def bump():
    counter = counter + 1
    return counter

print(bump())


UnboundLocalError: cannot access local variable 'counter' where it is not associated with a value

`cannot access local variable 'counter' where it is not associated with a value`.

Read it with the rule in mind. Because `bump` assigns to `counter`, Python treats `counter` as
**local throughout the function**. The right-hand side then reads a local that has not been
given a value yet.

The global `counter` is untouched and irrelevant. Python never considered it.


In [5]:
print(counter)     # unchanged


0


### global, and why to avoid it

`global` tells Python that a name refers to the outer one.


In [6]:
tally = 0

def bump():
    global tally
    tally = tally + 1
    return tally

print(bump(), bump(), bump())
print("tally is now", tally)


1 2 3
tally is now 3


It works, and it is rarely the right answer. A function that changes outside state gives a
different result depending on what ran before it, which makes it hard to reason about and hard
to test.

Returning a value is almost always better:


In [7]:
def bumped(n):
    return n + 1

tally = 0
tally = bumped(tally)
tally = bumped(tally)

print(tally)


2


Now the function depends on nothing but its argument, and the change is visible at the call
site rather than hidden inside.

### Mutating against rebinding

This is the section that answers "why did my list change" and "why did my list not change",
which are the same question.


In [8]:
def add_to(items):
    items.append("added")      # changes the list itself

def replace(items):
    items = ["replaced"]       # points the local name somewhere else

basket = ["original"]

add_to(basket)
print("after add_to: ", basket)

replace(basket)
print("after replace:", basket)


after add_to:  ['original', 'added']
after replace: ['original', 'added']


`add_to` changed the caller's list. `replace` did not, and never could.

The **Values and Variables** notebook explains why. The parameter `items` is a **second label on
the same list**. `items.append(...)` changes the list both labels point at, so the caller sees
it. `items = [...]` moves only the local label onto a new list, and the caller's label is
untouched.

The practical rule: a function can change a **mutable** value you pass in, and can never change
which value your name refers to.

Immutable values sidestep the question entirely, because nothing can change them.


In [9]:
def try_to_change(text):
    text = text.upper()
    return text

original = "hello"
returned = try_to_change(original)

print("original:", original)
print("returned:", returned)


original: hello
returned: HELLO


Strings are immutable, so `text.upper()` built a new one. Without the `return`, the work would
have been lost entirely.


### Shadowing

A local name hides an outer one of the same name for the length of the call.


In [10]:
name = "outer"

def show():
    name = "inner"
    return name

print(show())
print(name)


inner
outer


No conflict and no error. Two different names that happen to look alike.

Shadowing becomes a problem when the name you hide is one of Python's own.


In [11]:
list = [1, 2, 3]

print(list)
print(list("abc"))


[1, 2, 3]


TypeError: 'list' object is not callable

`'list' object is not callable`. The name `list` now refers to your list, so `list("abc")` tries
to call a list, which is not a thing you can do.

The built-in is not gone; it is hidden. Deleting your name uncovers it again.


In [12]:
del list

print(list("abc"))


['a', 'b', 'c']


Names worth avoiding for exactly this reason: `list`, `dict`, `set`, `str`, `int`, `sum`, `min`,
`max`, `type`, `id`, `input`. `items`, `values`, `counts` and `records` are all safe and usually
more descriptive anyway.


### Functions inside functions

A function defined inside another can see the outer function's names.


In [13]:
def outer():
    message = "from outer"

    def inner():
        return message      # found in the enclosing scope

    return inner()

print(outer())


from outer


That is the **enclosing** step in the lookup order, sitting between local and global.

Assignment behaves the same way it always does: it stays local unless you say otherwise. For an
enclosing scope, the keyword is `nonlocal`.


In [14]:
def without_nonlocal():
    n = 0
    def step():
        n = 99          # a brand new local inside step
    step()
    return n

def with_nonlocal():
    n = 0
    def step():
        nonlocal n
        n += 1
    step()
    step()
    return n

print(without_nonlocal())
print(with_nonlocal())


0
2


### Closures

A function can be **returned** from another, and it keeps access to the names it was created
with, even after the outer call has finished.


In [15]:
def multiplier(factor):
    def multiply(n):
        return n * factor
    return multiply

double = multiplier(2)
triple = multiplier(3)

print(double(10), triple(10))
print(double(5), triple(5))


20 30
10 15


`multiplier(2)` has returned, and its `factor` should be gone. It is not: `double` carries it.
A function bundled with the names it captured is called a **closure**.

Each call to `multiplier` makes a separate one, which is why `double` and `triple` do not
interfere.

This is what makes `key=lambda w: w[-1]` work in the **Functions** notebook, and it is behind a
great deal of library code that hands you a configured function.


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/15-scope-solutions.ipynb).

**1.** Write a function that creates a name inside itself, then try to print that name after
calling it. Predict what happens before you run it.


In [16]:
# your code here


**2.** Create `rate = 0.1` outside a function, then write a function that reads it and returns
`100 * rate`. No parameters.


In [17]:
# your code here


**3.** Rewrite task 2 so that `rate` is a parameter with a default of `0.1`. Say in a comment
which version you would rather maintain.


In [18]:
# your code here


**4.** Write `add_zero(items)` that appends `0` to the list it is given. Call it on a list you
made outside, then print that list to show it changed.


In [19]:
# your code here


**5.** Write `clear(items)` that sets `items = []` inside the function. Call it on a list from
outside and print the list afterward. Explain in a comment why it still has its contents.


In [20]:
# your code here


**6.** Write `adder(n)` that returns a function adding `n` to its argument. Make `add_five` from
it and print `add_five(10)`.


In [21]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### UnboundLocalError: read before assigned, in the same function

The error this notebook exists to explain.


In [22]:
total = 100

def discount():
    total = total * 0.9
    return total

print(discount())


UnboundLocalError: cannot access local variable 'total' where it is not associated with a value

The assignment on the first line makes `total` local for the whole function, so the right-hand
side is reading a local that has no value yet. The outer `total` was never in the running.

Three fixes, in order of preference:


In [23]:
def discount(total):            # best: take it as an argument
    return total * 0.9

print(discount(100))

total = 100
discounted = discount(total)    # the caller decides what to do with the result
print(total, discounted)


90.0
100 90.0


### NameError: a local name outside its function


In [24]:
def compute():
    working = 42
    return "done"

compute()
print(working)


NameError: name 'working' is not defined

`name 'working' is not defined`. The name existed only during the call. If you need the value,
`return` it.


### TypeError: a built-in name that was reused

Covered above, and worth seeing in its usual form: the error appears far from the line that
caused it.


In [25]:
sum = 0
for n in [1, 2, 3]:
    sum = sum + n

print(sum)
print(sum([4, 5, 6]))


6


TypeError: 'int' object is not callable

The loop is fine and prints `6`. The failure comes later, when something tries to use `sum` as
the built-in it used to be.

In a notebook this is especially easy to hit, because the name survives in memory until the
runtime is restarted. Name the variable `total` and the problem never arises.


In [26]:
del sum

total = 0
for n in [1, 2, 3]:
    total = total + n

print(total, sum([4, 5, 6]))


6 15


### The quiet one: a closure captures the name, not the value

This does not raise. It gives the same answer three times.


In [27]:
funcs = [lambda: i for i in range(3)]

print([f() for f in funcs])


[2, 2, 2]


Three functions were made, and every one returns `2`.

Each `lambda` captured the **name** `i`, not the value `i` had at the time. By the time any of
them ran, the comprehension had finished and `i` was left at `2`.

Bind the value at definition time with a default argument, which is evaluated immediately:


In [28]:
funcs = [lambda i=i: i for i in range(3)]

print([f() for f in funcs])


[0, 1, 2]


This is rare in everyday code and worth recognizing when it appears. It shows up whenever
functions are built inside a loop.


## Recap

- A name assigned inside a function is **local** and disappears when the call ends.
- Lookup searches outward: local, enclosing, global, built-in. The first match wins.
- Reading an outer name works. **Assigning always stays local**, which is what causes
  `UnboundLocalError`.
- `global` and `nonlocal` override that, and returning a value is almost always better.
- A function can **mutate** a value passed to it, and can never **rebind** the caller's name.
- Reusing a built-in name hides it, and the error appears wherever the built-in was next needed.
- A **closure** is a function that keeps the names it was created with, which is how
  `key=lambda` works.


## What is next

The **Errors and Exceptions** notebook, which stops treating errors as the end of the run. Every
notebook so far has shown tracebacks on purpose; this is where you learn to read one properly,
catch what you expect, and raise your own.


---

&#8592; **Previous:** [Functions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/14-functions.ipynb)  &nbsp;·&nbsp;  [Python from the Start Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)  &nbsp;·&nbsp;  **Next:** [Errors and Exceptions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/16-errors-and-exceptions.ipynb) &#8594;
